# Student Performance Prediction and Analysis

**Goal:** estimate exam scores (regression) and predict pass/fail (classification).

> **Data note:** the included CSV is synthetic/simulated for learning. It is not real student data.

This notebook performs data inspection, visual exploration, train/test splitting, model training, evaluation, and interpretation.

In [ ]:
from pathlib import Path
DATA_PATH = Path("data/student_performance.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    # Self-contained fallback for running this notebook on GitHub/Colab without the CSV file.
    rng = np.random.default_rng(42)
    n = 500
    study = np.clip(rng.normal(3.8, 1.7, n), 0.2, 10)
    attendance = np.clip(rng.normal(78, 12, n), 35, 100)
    previous = np.clip(rng.normal(65, 14, n), 20, 100)
    sleep = np.clip(rng.normal(7, 1.1, n), 3.5, 10)
    assignments = np.clip(rng.normal(75, 17, n), 10, 100)
    noise = rng.normal(0, 9, n)
    score = np.clip(-15 + 3.0*study + 0.18*attendance + 0.22*previous
                    + sleep + 0.10*assignments + noise, 0, 100)
    df = pd.DataFrame({
        "study_hours_per_day": study.round(1),
        "attendance_percent": attendance.round(1),
        "previous_exam_score": previous.round(1),
        "sleep_hours": sleep.round(1),
        "assignment_completion_percent": assignments.round(1),
        "final_exam_score": score.round(1),
        "passed": (score >= 40).astype(int)
    })
print("Dataset source:", "CSV file" if DATA_PATH.exists() else "reproducible synthetic data generated in notebook")
df.head()

In [ ]:
print("Shape:", df.shape)
print("\\nMissing values per column:")
print(df.isna().sum())
print("\\nSummary statistics:")
display(df.describe().round(2))

In [ ]:
features = ["study_hours_per_day", "attendance_percent", "previous_exam_score", "sleep_hours", "assignment_completion_percent"]
X = df[features]
y_score = df["final_exam_score"]
y_pass = df["passed"]

df[features + ["final_exam_score"]].hist(figsize=(12, 8), bins=18)
plt.suptitle("Distribution of Student Performance Variables")
plt.tight_layout()
plt.show()

## 1. Regression: predict the final exam score

We compare Linear Regression (simple, interpretable baseline) with Random Forest Regressor (can capture nonlinear relationships). Evaluation metrics: MAE, RMSE, and R². Lower MAE/RMSE and higher R² are preferred.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_score, test_size=0.2, random_state=42)
regressors = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=150, random_state=42, min_samples_leaf=3)
}
reg_rows, reg_predictions = [], {}
for name, model in regressors.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    reg_predictions[name] = pred
    reg_rows.append({"Model": name, "MAE": mean_absolute_error(y_test, pred),
                     "RMSE": np.sqrt(mean_squared_error(y_test, pred)), "R2": r2_score(y_test, pred)})
reg_results = pd.DataFrame(reg_rows).sort_values("RMSE")
display(reg_results.round(3))

In [ ]:
best_reg_name = reg_results.iloc[0]["Model"]
plt.figure(figsize=(7,5))
plt.scatter(y_test, reg_predictions[best_reg_name], alpha=0.75)
plt.plot([0,100],[0,100], linestyle="--")
plt.xlabel("Actual score"); plt.ylabel("Predicted score")
plt.title(f"Actual vs Predicted — {best_reg_name}")
plt.tight_layout(); plt.show()

## 2. Classification: predict pass/fail

A score of 40 or more is defined as a pass for this demonstration. Logistic Regression is an interpretable baseline; Random Forest Classifier is a nonlinear comparison. We report accuracy, precision, recall, and F1-score.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, y_pass, test_size=0.2, random_state=42, stratify=y_pass)
classifiers = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=150, random_state=42, min_samples_leaf=3)
}
cls_rows, cls_models = [], {}
for name, model in classifiers.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    cls_models[name] = model
    cls_rows.append({"Model": name, "Accuracy": accuracy_score(yc_test,pred),
                     "Precision": precision_score(yc_test,pred,zero_division=0),
                     "Recall": recall_score(yc_test,pred,zero_division=0),
                     "F1": f1_score(yc_test,pred,zero_division=0)})
cls_results = pd.DataFrame(cls_rows).sort_values("F1", ascending=False)
display(cls_results.round(3))

In [ ]:
best_cls_name = cls_results.iloc[0]["Model"]
ConfusionMatrixDisplay.from_estimator(cls_models[best_cls_name], Xc_test, yc_test)
plt.title(f"Confusion Matrix — {best_cls_name}")
plt.tight_layout(); plt.show()

In [ ]:
rf = regressors["Random Forest Regressor"]
importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", figsize=(7,5))
plt.xlabel("Feature importance")
plt.title("Random Forest Regressor — Feature Importance")
plt.tight_layout(); plt.show()
display(importance.sort_values(ascending=False).rename("importance").to_frame())

## Conclusion and limitations

Compare the held-out metrics above and identify the strongest model by the metric appropriate to the task. Regression estimates a numeric score; classification predicts a pass/fail category.

The data in this repository is simulated, so the scores demonstrate a reproducible modelling workflow rather than proving real-world accuracy. Real deployment would require representative, consented student data, checks for bias, and careful communication that predictions are uncertain.